In [ ]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

def resolve_path(path):
    candidate = Path(path)
    if candidate.exists():
        return candidate
    text = str(path).replace('\\', '/')
    name = Path(text).name
    special = {
        'positive_controls.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'negative_controls.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'Ten_positive_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'Ten_negative_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'fcg.txt': DATA_DIR / 'fcg.txt',
    }
    if name in special:
        return special[name]
    matches = [p for p in PROJECT_ROOT.rglob(name) if '.ipynb_checkpoints' not in p.parts and '.git' not in p.parts]
    if len(matches) == 1:
        return matches[0]
    if not candidate.suffix and (candidate.is_absolute() or ':' in text):
        return PROJECT_ROOT
    return candidate

from pdm_learn.preprocessing import build_density_map, density_centers, densitymap, drop_nan, extract, mut_trim, normalize, trim, trim_pairs
from pdm_learn.modeling import KFold_PR, LOOCV, LOOCV_grouped_plot, area_table, core_predict, heatmap, importance_test, ks_pvalue
from pdm_learn.simulation import eps, partition


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LogisticRegression
from itertools import compress
from scipy import stats
import random
from bisect import bisect
import warnings
import xgboost as xgb
import time

In [ ]:
prot_int = pd.read_table(DATA_DIR / 'fcg.txt')
mask = prot_int['Organism'].str.contains('Human')
prot_int_human = prot_int[mask].reset_index(drop=True)
subunits = prot_int_human.iloc[:,-7].to_numpy()
subunits

In [ ]:
used_gene = np.array([])
times_used = np.array([])
for str in subunits:
    for gene in str.strip().split(';'):
        if gene not in used_gene:
            num = 0
            for str2 in subunits:
                if gene in str2:
                    num+=1
            used_gene = np.append(used_gene, gene)
            times_used = np.append(times_used, num)

plt.hist(times_used, bins=np.arange(times_used.min(), times_used.max()+1))
plt.show()
pd.DataFrame({'Gene':used_gene, 'Times Used':times_used.astype(int)})

In [ ]:
num_gene = np.array([])
for str in subunits:
    arr = str.strip().split(';')
    used = []
    num = 0
    for i in arr:
        if i not in used:
            used.append(i)
            num+=1
    num_gene = np.append(num_gene, num)

plt.hist(num_gene, bins=np.arange(num_gene.min(), num_gene.max()+1))
plt.show()
pd.DataFrame({'Functional Group':prot_int_human.iloc[:,0].to_numpy(), 'Gene Number':num_gene.astype(int)})

In [ ]:
used = np.array([])
positive = np.array([[None,None]])
for str in subunits:
    genes = str.strip().split(';')
    temp = None
    for gene in genes:
        if gene not in used and gene != temp:
            if temp == None:
                temp = gene
            else:
                used = np.append(used, [temp,gene])
                positive = np.concatenate((positive, [[temp, gene]]))
                temp = None
    temp = None

positive = np.delete(positive, 0, 0)
positive = [f'{p1}.{p2}' for p1, p2 in positive]
np.savetxt(DATA_DIR / 'positive_pairs.csv', positive, delimiter=",", fmt='%s')

In [ ]:
used = np.array([])
negative = np.array([[None,None]])
for n in range(len(subunits)):
    genes = subunits[n].strip().split(';')
    for gene in genes:
        done = False
        if gene not in used:
            for m in range(n+1,len(subunits)):
                match = subunits[m].strip().split(';')
                if gene not in match:
                    for temp in match:
                        if temp not in used:
                            used = np.append(used, [gene, temp])
                            negative = np.concatenate((negative, [[gene, temp]]))
                            done = True
                            break
                if done:
                    break

negative = np.delete(negative, 0, 0)
negative

In [ ]:
data = pd.read_csv(DATA_DIR / 'Trimmed data' / 'dataset_trimmed_v3.csv').dropna().reset_index(drop=True)

oncogenes = pd.read_csv(DATA_DIR / 'oncogene.txt').to_numpy().flatten()
oncogenes = np.append(oncogenes,'MYCL')
oncogenes = np.array(list(set(oncogenes).intersection(data.iloc[:,0])))

In [ ]:
input_data = data.copy().sample(frac=1, random_state=1, ignore_index=True)
input_data['oncogene'] = np.isin(input_data.iloc[:,0], oncogenes)

# remove specific columns
# drop_index = input_data.columns[['gene_exp' in i for i in input_data.columns]]
# input_data.drop(drop_index, axis=1, inplace=True)

# split into positive and negative
positive = input_data.iloc[list(compress(range(len(input_data)), input_data.iloc[:,-1]))].reset_index(drop=True)
negative = input_data.drop(list(compress(range(len(input_data)), input_data.iloc[:,-1]))).reset_index(drop=True)
input_data

In [ ]:
scores = core_predict(positive.iloc[:,1:-1].values, negative.iloc[:,1:-1].values, 100, model='GBR', ks_test=True, features_left=None)
result = pd.concat([negative.iloc[:,0], pd.Series(scores)], axis=1)
result = result.sort_values(by=result.columns[-1], ascending=False)
# result.to_csv(ARTIFACTS_DIR / 'results' / 'oncogene_ranking.csv', index=False)
result

In [ ]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [ ]:
curr = time.time()
lr_area, lr_x, lr_y = LOOCV(positive.iloc[:,1:-1].values, negative.iloc[:,1:-1].values, 25, 
      model='LR', ks_test=True, features_left=250, graph=True, equation=True)
lr_time = time.time()-curr

In [ ]:
plt.plot(svm_x, svm_y, label = "SVM ("+str(round(svm_area, 3))+")"+str(svm_time))
plt.plot(gbr_x, gbr_y, label = "GBT ("+str(round(gbr_area, 3))+")"+str(gbr_time), linestyle="-")
plt.plot(xgb_x, xgb_y, label = "XGB ("+str(round(xgb_area, 3))+")"+str(xgb_time), linestyle=":")
plt.plot(dtr_x, dtr_y, label = "DTR ("+str(round(dtr_area, 3))+")"+str(dtr_time), linestyle="-.")
plt.plot(lr_x, lr_y, label = "LR ("+str(round(lr_area, 3))+")"+str(lr_time), linestyle=" ")
plt.title("AUC Curves")
plt.legend()
plt.show()

In [ ]:
graphs = [method_area, method_x, method_y, pearson_area, pearson_x, pearson_y, spearman_area, spearman_x, spearman_y,]
pd.Series(graphs).to_csv(DATA_DIR / 'Trimmed data' / 'graphs.csv', index=False)

In [ ]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [ ]:
feature_arr = [10, 20, 50, 100, 124, 150, 200, 250, 300, 349]
areas = area_table(positive.iloc[:,1:-1].values, negative.iloc[:,1:-1].values, 20, 
                       model='GBR', feat_arr=feature_arr)

In [ ]:
input_data = data.copy().sample(frac=1, random_state=1, ignore_index=True)
input_data['oncogene'] = np.isin(input_data.iloc[:,0], oncogenes)

# remove specific columns
drop_index = input_data.columns[['CRISPR' in i for i in input_data.columns]]
input_data.drop(drop_index, axis=1, inplace=True)
drop_index = input_data.columns[['copy_num' in i for i in input_data.columns]]
input_data.drop(drop_index, axis=1, inplace=True)
drop_index = input_data.columns[['gene_exp' in i for i in input_data.columns]]
input_data.drop(drop_index, axis=1, inplace=True)

# split into positive and negative
positive = input_data.iloc[list(compress(range(len(input_data)), input_data.iloc[:,-1]))].reset_index(drop=True)
negative = input_data.drop(list(compress(range(len(input_data)), input_data.iloc[:,-1]))).reset_index(drop=True)

feature_arr = [1, 2, 5, 7, 10, 13]
areas = area_table(positive.iloc[:,1:-1].values, negative.iloc[:,1:-1].values, 20, 
                       model='GBR', feat_arr=feature_arr)
print(areas)

In [ ]:
feature_area = np.array([feature_arr, areas])

In [ ]:
plt.scatter([10, 20, 30, 40, 60, 80, 100, 125, 150, 185], [0.8522559907229497, 0.8368654827705877, 0.8380526235599834, 0.8311466806612438, 0.836799883190425, 0.8432476870857704, 0.8485379758085677, 0.8410987718065701, 0.8428678443554736, 0.8368633666550985])
plt.title('GBR')
plt.show()

In [ ]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [ ]:
p_val = ks_pvalue(positive.iloc[:,1:-1].values, negative.iloc[:,1:-1].values)

In [ ]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [ ]:
log_p_val = np.log10(p_val)
heatmap(log_p_val, [[7,7],[7,7],[7,2],[7,7],[7,7],[7,2],[7,7],[7,2],[7,7],[7,2]], 
        cmap='gist_heat_r', min=np.min(log_p_val), max=np.max(log_p_val), flip=True, axes=False, colorbar=False)

In [ ]:
from sklearn.inspection import permutation_importance

In [ ]:
# Shared helper functions now live in src/pdm_learn.
# See the project setup cell at the top of this notebook for imports.


In [ ]:
importance_test(positive, negative, 6, True)

In [ ]:
temp = pd.DataFrame()
for i in range(3):
    temp = pd.concat([temp, pd.DataFrame({'feature ' + str(i):np.random.normal(size=100)})], axis=1)
temp = pd.concat([temp, pd.DataFrame({'Result':temp.iloc[:,0]+temp.iloc[:,1]})], axis=1)
temp

In [ ]:
model.predict([[1,1,0], [0,1,0], [0,1,0], [0,1,0], [0,1,0], [0,1,0], [0,1,0], [0,1,0]])

In [ ]:
X = temp.drop(['Result'], axis=1)
y = temp['Result']

model = SVR().fit(X, y)
results = permutation_importance(model, X, y, n_repeats=20)
# get importance
importance = results.importances_mean
# plot feature importance
plt.bar([x for x in range(len(importance))], importance)
plt.show()